In [ ]:
import sys, os
import rasterio

import geopandas as gpd
import GOSTrocks.rasterMisc as rMisc

from shapely.geometry import Polygon


In [ ]:
base_folder = r"C:\WBG\Work\Projects\SSD_Health\SSD_Kamer_Imagery"
in_admin = os.path.join(base_folder, "SDN_sel_adm3.geojson")

global_ghsl = "C:/WBG/Work/data/GHSL/GHS_BUILT_S_E2025_GLOBE_R2023A_54009_100_V1_0.tif"

local_ghsl = os.path.join(base_folder, "SDN_GHSL_BUILT.tif")

In [ ]:
inA = gpd.read_file(in_admin)
inR = rasterio.open(global_ghsl)

if inA.crs != inR.crs:
    inA = inA.to_crs(inR.crs)

In [ ]:
# clip raster by ADMIN area extent
if not os.path.exists(local_ghsl):
    rMisc.clipRaster(inR, inA, local_ghsl, crop=True)

curR = rasterio.open(local_ghsl)

# thgreshold GHSL built area and convert to vector
ghsl_thresh = 1000
ghsl_data = curR.read(1)
ghsl_data[ghsl_data == curR.meta['nodata']] = 0
ghsl_mask = ghsl_data >= ghsl_thresh

with rMisc.create_rasterio_inmemory(curR.meta.copy(), ghsl_mask.astype(rasterio.uint8)) as ghsl_mask_raster:
    ghsl_built_areas = rMisc.vectorize_raster(ghsl_mask_raster)
    
ghsl_built_areas = ghsl_built_areas[ghsl_built_areas['value'] == 1].copy()
ghsl_built_areas = ghsl_built_areas.to_crs(20135)  # UTM 35N
ghsl_built_areas["area_m2"] = ghsl_built_areas.geometry.area

ghsl_built_areas['geometry'] = ghsl_built_areas['geometry'].buffer(5000)
if inA.crs != ghsl_built_areas.crs:
    inA = inA.to_crs(ghsl_built_areas.crs)
all_shapes = ghsl_built_areas.union_all().intersection(inA.union_all())
all_geoms = [Polygon(x.exterior.coords) for x in all_shapes.geoms]
ghsl_dissolved = gpd.GeoDataFrame(geometry=all_geoms, crs=ghsl_built_areas.crs)
ghsl_dissolved['IDX'] = ghsl_dissolved.index + 1
ghsl_dissolved['area_m2'] = ghsl_dissolved.geometry.area
ghsl_dissolved.to_file(os.path.join(base_folder, "SDN_GHSL_BUILT_5km_buffer.geojson"), driver="GeoJSON")


In [ ]:
ghsl_dissolved['area_m2'].sum()/1000000


In [ ]:
ghsl_built_areas

In [ ]:
curR.meta